In [21]:
import os
from pathlib import Path

print("Jupyter's current folder:", os.getcwd())
print("Items in this folder:", os.listdir())
print("Matching Excel files found nearby:", list(Path(".").rglob("SoundTracker_Recording_Names.xlsx")))

Jupyter's current folder: c:\Users\charl\OneDrive\Dokumente\GitHub\MDM3-Mindear-Project
Items in this folder: ['.git', 'all_phone_configurations.png', 'all_predicted_sound_maps.png', 'Basic_Sound_Map_Files', 'diffusion_lms_model.py', 'Diffusion_lms_model_testing.ipynb', 'extracted_kit', 'gp_kernel_and_predict.py', 'gp_model.ipynb', 'gp_single_source.py', 'gp_tast.py', 'Ordinary_Kriging_Model_buffer.py', 'Ordinary_Kriging_Model_buffer_diffusion.py', 'Ordinary_Kriging_Model_no_buffer.py', 'Ordinary_Kriging_Model_Test_Runs.ipynb', 'process_soundtrackr_calibration.py', 'README.md', 'README_MDM3_updated.md', 'SoundTracker_Calibration_Kit', 'SoundTracker_Calibration_Kit.zip', 'test_kit_run', 'Working_final_maps', '__pycache__']
Matching Excel files found nearby: [WindowsPath('extracted_kit/SoundTracker_Calibration_Kit/SoundTracker_Recording_Names.xlsx'), WindowsPath('test_kit_run/SoundTracker_Calibration_Kit/SoundTracker_Recording_Names.xlsx')]


In [22]:
import zipfile

with zipfile.ZipFile("SoundTracker_Calibration_Kit.zip", "r") as z:
    z.extractall(".")

In [23]:
#!/usr/bin/env python3
"""
SoundTrackr Smartphone Calibration Data Organiser & Error Analysis Pipeline
---------------------------------------------------------------------------
Implements the experimental protocol and statistical methodology defined in:
  1. Charlie Experiment Variances.pdf (Section 3: Smartphone Measurement Error Modelling)
  2. SoundTrackr Prototype Tutorial.docx (Collector CSV export format & cleaning rules)
  3. SoundTracker_Calibration_Kit folder structure & README.txt
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats


# ==============================================================================
# CONFIGURATION CONSTANTS
# ==============================================================================
# Collector browser AudioWorklet clamps levels outside [30.0, 120.0] dB
CLAMP_MIN_DB = 30.0
CLAMP_MAX_DB = 120.0

# Protocol specifies a consistent 15-second analysis window per trial (README.txt).
# Set to None to use all valid unclamped rows in each recording.
TARGET_WINDOW_SECONDS: Optional[int] = 15

# Default random seed for reproducible simulated measurement error generation
DEFAULT_RANDOM_SEED = 42


# ==============================================================================
# CORE ACOUSTIC & STATISTICAL FUNCTIONS
# ==============================================================================
def energy_mean_db(
    levels_db: pd.Series | np.ndarray,
    durations_s: Optional[pd.Series | np.ndarray] = None,
) -> float:
    """
    Calculate the equivalent continuous sound level (Leq) via energy averaging.
    Never averages decibels arithmetically.

    Formula (Charlie Experiment Variances, Eq. 1):
        L_phone,i = 10 * log10( (1 / N_i) * sum( 10^(L_ij / 10) ) )
    Supports duration weighting if integration intervals differ.
    """
    levels = np.asarray(levels_db, dtype=float)
    valid_mask = np.isfinite(levels)
    levels = levels[valid_mask]

    if levels.size == 0:
        return np.nan

    linear_energy = np.power(10.0, levels / 10.0)

    if durations_s is not None:
        durations = np.asarray(durations_s, dtype=float)[valid_mask]
        if np.all(np.isfinite(durations)) and np.sum(durations) > 0:
            mean_energy = np.sum(durations * linear_energy) / np.sum(durations)
            return float(10.0 * np.log10(mean_energy))

    return float(10.0 * np.log10(np.mean(linear_energy)))


def apply_measurement_error(
    l_true: np.ndarray | pd.Series | float,
    mu: float,
    sigma: float,
    seed: int = DEFAULT_RANDOM_SEED,
) -> np.ndarray:
    """
    Introduce experimentally informed measurement uncertainty into simulated
    phone readings (Charlie Experiment Variances, Section 3.4, Eqs. 6-7):
        L_measured,i = L_true,i + mu + eps_i,   eps_i ~ N(0, sigma^2)
    """
    rng = np.random.default_rng(seed)
    true_arr = np.asarray(l_true, dtype=float)
    epsilon = rng.normal(loc=0.0, scale=sigma, size=true_arr.shape)
    return true_arr + mu + epsilon


# ==============================================================================
# DATA LOADING, CLEANING & ORGANISATION
# ==============================================================================
def load_experiment_manifest(kit_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Load the 180-trial naming schedule and phone ID key from
    SoundTracker_Recording_Names.xlsx.
    """
    xlsx_path = kit_root / "SoundTracker_Recording_Names.xlsx"
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Could not find manifest workbook at: {xlsx_path}")

    recordings_df = pd.read_excel(xlsx_path, sheet_name="Recording Names")
    phone_key_df = pd.read_excel(xlsx_path, sheet_name="Phone ID Key")

    # Derive condition folder prefix directly from expected filename (e.g. 'E1_65dB')
    recordings_df["Condition_Folder"] = (
        recordings_df["Filename (copy to rename)"]
        .astype(str)
        .str.extract(r"^([^_]+_[^_]+)")[0]
    )
    recordings_df["Expected_Rel_Path"] = recordings_df.apply(
        lambda r: str(
            Path("01_Raw_Data")
            / r["Condition_Folder"]
            / r["Phone ID"]
            / r["Filename (copy to rename)"]
        ),
        axis=1,
    )
    return recordings_df, phone_key_df


def load_or_create_reference_log(
    kit_root: Path, recordings_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Load 04_Reference_Meter_Exports/Reference_Log.csv if present; otherwise
    create a pre-populated template matching all 180 trials so reference SLM
    levels and optional multi-trial slice timestamps can be logged cleanly.
    """
    ref_dir = kit_root / "04_Reference_Meter_Exports"
    ref_dir.mkdir(parents=True, exist_ok=True)
    ref_log_path = ref_dir / "Reference_Log.csv"

    if ref_log_path.exists():
        ref_df = pd.read_csv(ref_log_path)
        for str_col in ["Source_Export_File", "Window_Start_UTC", "Window_End_UTC", "Notes"]:
            if str_col in ref_df.columns:
                ref_df[str_col] = ref_df[str_col].fillna("").astype(str)
    else:
        ref_df = pd.DataFrame(
            {
                "Filename": recordings_df["Filename (copy to rename)"],
                "Condition_Folder": recordings_df["Condition_Folder"],
                "Experiment": recordings_df["Experiment"],
                "Target_Level_dBA": recordings_df["Target level (dBA)"],
                "Orientation": recordings_df["Orientation"],
                "Pocket": recordings_df["Pocket"],
                "Phone_ID": recordings_df["Phone ID"],
                "Trial": recordings_df["Trial"],
                # Enter actual calibrated SLM reading (LAeq) OR before/after block readings:
                "Ref_LAeq_dB": np.nan,
                "Ref_Before_dB": np.nan,
                "Ref_After_dB": np.nan,
                # Optional: populate only if multiple trials were exported in one raw CSV
                "Source_Export_File": "",
                "Window_Start_UTC": "",
                "Window_End_UTC": "",
                "Notes": "",
            }
        )
        ref_df.to_csv(ref_log_path, index=False)
        print(f"[INFO] Created template reference log at: {ref_log_path}")

    # Resolve L_reference,i: use direct Ref_LAeq_dB if given, else mean(Ref_Before_dB, Ref_After_dB)
    before_after_mean = ref_df[["Ref_Before_dB", "Ref_After_dB"]].mean(axis=1)
    ref_df["Resolved_Ref_LAeq_dB"] = ref_df["Ref_LAeq_dB"].fillna(before_after_mean)
    return ref_df


def locate_raw_trial_csv(
    kit_root: Path,
    expected_rel_path: str,
    condition_folder: str,
    phone_id: str,
    source_export_file: str = "",
) -> Optional[Path]:
    """
    Locate the raw phone CSV export without modifying 01_Raw_Data/.
    Checks:
      1. Exact expected path: 01_Raw_Data/<Condition>/<Phone_ID>/<Filename>.csv
      2. Explicit multi-trial Source_Export_File (if specified in Reference_Log.csv)
    """
    exact_path = kit_root / expected_rel_path
    if exact_path.exists():
        return exact_path

    if isinstance(source_export_file, str) and source_export_file.strip():
        candidate_paths = [
            kit_root / source_export_file.strip(),
            kit_root / "01_Raw_Data" / source_export_file.strip(),
            kit_root / "01_Raw_Data" / condition_folder / phone_id / source_export_file.strip(),
            kit_root / "01_Raw_Data" / condition_folder / source_export_file.strip(),
        ]
        for p in candidate_paths:
            if p.exists():
                return p

    return None


def clean_and_extract_trial(
    raw_csv_path: Path,
    window_start_utc: str = "",
    window_end_utc: str = "",
    target_window_seconds: Optional[int] = TARGET_WINDOW_SECONDS,
) -> Tuple[pd.DataFrame, dict]:
    """
    Read a SoundTrackr Collector CSV export, apply protocol cleaning rules
    (SoundTrackr Prototype Tutorial Section 4), and return the cleaned 1-second
    trial DataFrame along with QC diagnostics.

    Expected Collector CSV columns:
      recorded_at, event_id, contributor_bucket, lat, lon, accuracy_m,
      laeq_db, lamax_db, integration_seconds, calibration_offset_db,
      uploaded, invite, phone_model, device_class, consent_version
    """
    df = pd.read_csv(raw_csv_path)
    if "laeq_db" not in df.columns:
        raise ValueError(f"Missing required 'laeq_db' column in {raw_csv_path}")

    # Parse ISO-8601 UTC window-end timestamps
    if "recorded_at" in df.columns:
        df["recorded_at_dt"] = pd.to_datetime(df["recorded_at"], utc=True, errors="coerce")
        df = df.sort_values("recorded_at_dt").reset_index(drop=True)

    def _is_valid_ts(val: object) -> bool:
        return (
            val is not None
            and pd.notna(val)
            and str(val).strip().lower() not in ("", "nan", "none", "nat")
        )

    # Optional timestamp slicing when one raw file holds multiple trials
    has_start = _is_valid_ts(window_start_utc)
    has_end = _is_valid_ts(window_end_utc)
    if (has_start or has_end) and "recorded_at_dt" in df.columns:
        if has_start:
            t_start = pd.to_datetime(str(window_start_utc).strip(), utc=True)
            df = df[df["recorded_at_dt"] >= t_start]
        if has_end:
            t_end = pd.to_datetime(str(window_end_utc).strip(), utc=True)
            df = df[df["recorded_at_dt"] <= t_end]
        df = df.reset_index(drop=True)

    n_raw_rows = len(df)

    # Drop boundary-clamped readings (<= 30 dB or >= 120 dB) and explicit clipped flags
    df["laeq_db"] = pd.to_numeric(df["laeq_db"], errors="coerce")
    clamped_mask = (df["laeq_db"] <= CLAMP_MIN_DB) | (df["laeq_db"] >= CLAMP_MAX_DB)
    if "clipped" in df.columns:
        clipped_flag = df["clipped"].astype(str).str.lower().isin(["true", "1", "yes"])
        clamped_mask = clamped_mask | clipped_flag

    n_clamped_dropped = int(clamped_mask.sum())
    clean_df = df[~clamped_mask & df["laeq_db"].notna()].copy().reset_index(drop=True)

    # Enforce uniform 15-second central analysis window if recording is longer
    if (
        target_window_seconds is not None
        and not (has_start and has_end)
        and len(clean_df) > target_window_seconds
    ):
        trim_start = (len(clean_df) - target_window_seconds) // 2
        clean_df = clean_df.iloc[trim_start : trim_start + target_window_seconds].reset_index(drop=True)

    # Extract device metadata from Collector export
    def _first_non_null(col: str) -> str:
        if col in clean_df.columns and clean_df[col].dropna().shape[0] > 0:
            return str(clean_df[col].dropna().iloc[0])
        if col in df.columns and df[col].dropna().shape[0] > 0:
            return str(df[col].dropna().iloc[0])
        return ""

    diagnostics = {
        "n_raw_rows": n_raw_rows,
        "n_clamped_dropped": n_clamped_dropped,
        "n_valid_rows": len(clean_df),
        "contributor_bucket": _first_non_null("contributor_bucket"),
        "phone_model": _first_non_null("phone_model"),
        "device_class": _first_non_null("device_class"),
        "calibration_offset_db": _first_non_null("calibration_offset_db"),
    }
    return clean_df.drop(columns=["recorded_at_dt"], errors="ignore"), diagnostics


# ==============================================================================
# MAIN PROCESSING & STATISTICAL ANALYSIS WORKFLOW
# ==============================================================================
def run_pipeline(kit_root: Path, random_seed: int = DEFAULT_RANDOM_SEED) -> dict:
    processed_dir = kit_root / "02_Processed_Data"
    analysis_dir = kit_root / "03_Analysis"
    processed_dir.mkdir(parents=True, exist_ok=True)
    analysis_dir.mkdir(parents=True, exist_ok=True)

    # 1. Load naming manifest and reference meter log
    recordings_df, phone_key_df = load_experiment_manifest(kit_root)
    ref_df = load_or_create_reference_log(kit_root, recordings_df)

    merged_schedule = pd.merge(
        recordings_df,
        ref_df[
            [
                "Filename",
                "Resolved_Ref_LAeq_dB",
                "Source_Export_File",
                "Window_Start_UTC",
                "Window_End_UTC",
                "Notes",
            ]
        ],
        left_on="Filename (copy to rename)",
        right_on="Filename",
        how="left",
    )

    # 2. Process all 180 scheduled recordings
    trial_records = []
    discovered_devices = {}

    for _, row in merged_schedule.iterrows():
        fname = row["Filename (copy to rename)"]
        cond_folder = row["Condition_Folder"]
        phone_id = row["Phone ID"]

        raw_path = locate_raw_trial_csv(
            kit_root=kit_root,
            expected_rel_path=row["Expected_Rel_Path"],
            condition_folder=cond_folder,
            phone_id=phone_id,
            source_export_file=row.get("Source_Export_File", ""),
        )

        record = {
            "Filename": fname,
            "Condition_Folder": cond_folder,
            "Experiment": row["Experiment"],
            "Target_Level_dBA": row["Target level (dBA)"],
            "Orientation": row["Orientation"],
            "Pocket": row["Pocket"],
            "Phone_ID": phone_id,
            "Trial": int(row["Trial"]),
            "Raw_File_Found": raw_path is not None,
            "N_Raw_Rows": 0,
            "N_Clamped_Dropped": 0,
            "N_Valid_Seconds": 0,
            "Phone_LAeq_dB": np.nan,
            "Reference_LAeq_dB": row["Resolved_Ref_LAeq_dB"],
            "Measurement_Error_dB": np.nan,
            "Contributor_Bucket": "",
            "Phone_Model": "",
            "Device_Class": "",
            "QC_Status": "Missing Raw File",
        }

        if raw_path is not None:
            clean_df, diag = clean_and_extract_trial(
                raw_csv_path=raw_path,
                window_start_utc=str(row.get("Window_Start_UTC", "") or ""),
                window_end_utc=str(row.get("Window_End_UTC", "") or ""),
                target_window_seconds=TARGET_WINDOW_SECONDS,
            )

            # Save cleaned trial extract to 02_Processed_Data/ (never modifying 01_Raw_Data/)
            extract_out_path = processed_dir / fname
            clean_df.to_csv(extract_out_path, index=False)

            durations = (
                clean_df["integration_seconds"]
                if "integration_seconds" in clean_df.columns
                else None
            )
            l_phone = energy_mean_db(clean_df["laeq_db"], durations_s=durations)
            l_ref = row["Resolved_Ref_LAeq_dB"]
            err = (l_phone - l_ref) if (np.isfinite(l_phone) and np.isfinite(l_ref)) else np.nan

            record.update(
                {
                    "N_Raw_Rows": diag["n_raw_rows"],
                    "N_Clamped_Dropped": diag["n_clamped_dropped"],
                    "N_Valid_Seconds": diag["n_valid_rows"],
                    "Phone_LAeq_dB": round(l_phone, 4) if np.isfinite(l_phone) else np.nan,
                    "Measurement_Error_dB": round(err, 4) if np.isfinite(err) else np.nan,
                    "Contributor_Bucket": diag["contributor_bucket"],
                    "Phone_Model": diag["phone_model"],
                    "Device_Class": diag["device_class"],
                    "QC_Status": (
                        "Valid"
                        if np.isfinite(err)
                        else ("Missing Reference LAeq" if np.isfinite(l_phone) else "All Rows Clamped/Invalid")
                    ),
                }
            )

            if phone_id not in discovered_devices and diag["phone_model"]:
                discovered_devices[phone_id] = {
                    "Phone_Model": diag["phone_model"],
                    "Contributor_Bucket": diag["contributor_bucket"],
                    "Device_Class": diag["device_class"],
                }

        trial_records.append(record)

    trials_df = pd.DataFrame(trial_records)

    # Save master trial table in 02_Processed_Data/
    master_trials_path = processed_dir / "All_Trials_Processed.csv"
    trials_df.to_csv(master_trials_path, index=False)

    # Save updated Device Register in 02_Processed_Data/
    device_reg = phone_key_df.copy()
    device_reg["Detected_Phone_Model"] = device_reg["Phone ID"].map(
        lambda pid: discovered_devices.get(pid, {}).get("Phone_Model", "")
    )
    device_reg["Contributor_Bucket"] = device_reg["Phone ID"].map(
        lambda pid: discovered_devices.get(pid, {}).get("Contributor_Bucket", "")
    )
    device_reg["Device_Class"] = device_reg["Phone ID"].map(
        lambda pid: discovered_devices.get(pid, {}).get("Device_Class", "")
    )
    device_reg.to_csv(processed_dir / "Device_Register_Updated.csv", index=False)

    # 3. Filter valid paired recordings (both phone LAeq and reference LAeq available)
    valid_df = trials_df[trials_df["Measurement_Error_dB"].notna()].copy()
    m_valid = len(valid_df)

    print(f"Total scheduled trials: {len(trials_df)}")
    print(f"Raw phone CSVs found:   {int(trials_df['Raw_File_Found'].sum())}")
    print(f"Valid paired trials:    {m_valid}")

    if m_valid < 2:
        print(
            "\n[ACTION REQUIRED] Fewer than 2 valid paired trials found.\n"
            "Ensure raw phone exports are saved under 01_Raw_Data/<Condition>/<Phone_ID>/\n"
            "and actual reference SLM readings are entered in:\n"
            f"  {kit_root / '04_Reference_Meter_Exports' / 'Reference_Log.csv'}"
        )
        return {"valid_count": m_valid}

    # 4. Overall Bias (mu) and Pooled Sample Variance (sigma^2) — Eqs. 3, 4, 5
    errors = valid_df["Measurement_Error_dB"].to_numpy(dtype=float)
    mu = float(np.mean(errors))
    sigma_sq = float(np.var(errors, ddof=1))  # Unbiased sample variance: 1 / (M - 1)
    sigma = float(np.sqrt(sigma_sq))

    # 5. Breakdowns by Condition, Phone, and Phone x Condition Repeatability (Section 3.6)
    by_condition = (
        valid_df.groupby(
            ["Condition_Folder", "Experiment", "Target_Level_dBA", "Orientation", "Pocket"],
            as_index=False,
        )
        .agg(
            N_Trials=("Measurement_Error_dB", "count"),
            Mean_Error_dB=("Measurement_Error_dB", "mean"),
            Variance_dB2=("Measurement_Error_dB", lambda s: np.var(s, ddof=1) if len(s) > 1 else np.nan),
            SD_dB=("Measurement_Error_dB", lambda s: np.std(s, ddof=1) if len(s) > 1 else np.nan),
            Min_Error_dB=("Measurement_Error_dB", "min"),
            Max_Error_dB=("Measurement_Error_dB", "max"),
        )
    )
    by_condition.to_csv(analysis_dir / "Summary_By_Condition.csv", index=False)

    by_phone = (
        valid_df.groupby(["Phone_ID"], as_index=False)
        .agg(
            N_Trials=("Measurement_Error_dB", "count"),
            Mean_Error_dB=("Measurement_Error_dB", "mean"),
            Variance_dB2=("Measurement_Error_dB", lambda s: np.var(s, ddof=1) if len(s) > 1 else np.nan),
            SD_dB=("Measurement_Error_dB", lambda s: np.std(s, ddof=1) if len(s) > 1 else np.nan),
        )
    )
    by_phone.to_csv(analysis_dir / "Summary_By_Phone.csv", index=False)

    # Repeatability per Phone x Condition cell (across the 5 repeated trials)
    repeatability_df = (
        valid_df.groupby(["Condition_Folder", "Phone_ID"], as_index=False)
        .agg(
            N_Repeats=("Measurement_Error_dB", "count"),
            Cell_Mean_Error_dB=("Measurement_Error_dB", "mean"),
            Repeatability_Var_dB2=(
                "Measurement_Error_dB",
                lambda s: np.var(s, ddof=1) if len(s) > 1 else np.nan,
            ),
            Repeatability_SD_dB=(
                "Measurement_Error_dB",
                lambda s: np.std(s, ddof=1) if len(s) > 1 else np.nan,
            ),
        )
    )
    repeatability_df.to_csv(analysis_dir / "Summary_Repeatability_By_Cell.csv", index=False)

    # Pooled pure within-cell repeatability variance (informational; already inside sigma^2)
    valid_cells = repeatability_df[repeatability_df["N_Repeats"] > 1]
    if not valid_cells.empty:
        dof_rep = (valid_cells["N_Repeats"] - 1).sum()
        pooled_rep_var = float(
            ((valid_cells["N_Repeats"] - 1) * valid_cells["Repeatability_Var_dB2"]).sum() / dof_rep
        )
        pooled_rep_sd = float(np.sqrt(pooled_rep_var))
    else:
        pooled_rep_var, pooled_rep_sd = np.nan, np.nan

    # Shapiro-Wilk normality check (Section 3.5.3)
    shapiro_w, shapiro_p = (
        stats.shapiro(errors) if 3 <= m_valid <= 5000 else (np.nan, np.nan)
    )

    # Save overall parameters JSON & CSV for direct import into simulation models
    overall_summary = {
        "valid_recordings_M": m_valid,
        "overall_bias_mu_db": round(mu, 4),
        "pooled_variance_sigma2_db2": round(sigma_sq, 4),
        "pooled_sd_sigma_db": round(sigma, 4),
        "pure_repeatability_variance_db2": round(pooled_rep_var, 4) if np.isfinite(pooled_rep_var) else None,
        "pure_repeatability_sd_db": round(pooled_rep_sd, 4) if np.isfinite(pooled_rep_sd) else None,
        "shapiro_wilk_W": round(float(shapiro_w), 4) if np.isfinite(shapiro_w) else None,
        "shapiro_wilk_p": round(float(shapiro_p), 4) if np.isfinite(shapiro_p) else None,
        "recommended_random_seed": random_seed,
    }
    with open(analysis_dir / "Overall_Error_Parameters.json", "w", encoding="utf-8") as f:
        json.dump(overall_summary, f, indent=2)
    pd.DataFrame([overall_summary]).to_csv(analysis_dir / "Overall_Error_Parameters.csv", index=False)

    # 6. Histogram of Pooled Measurement Errors vs. Fitted Normal Distribution (Section 3.6)
    fig, ax = plt.subplots(figsize=(8.5, 5.2), dpi=150)
    n_bins = min(20, max(8, int(np.sqrt(m_valid))))
    ax.hist(
        errors,
        bins=n_bins,
        density=True,
        alpha=0.65,
        color="#2b6cb0",
        edgecolor="white",
        label=f"Pooled Experimental Errors (M = {m_valid})",
    )

    x_min = min(errors.min(), mu - 3.5 * sigma)
    x_max = max(errors.max(), mu + 3.5 * sigma)
    x_grid = np.linspace(x_min, x_max, 300)
    normal_pdf = stats.norm.pdf(x_grid, loc=mu, scale=sigma)
    ax.plot(
        x_grid,
        normal_pdf,
        color="#c53030",
        linewidth=2.2,
        label=rf"Fitted Normal $\mathcal{{N}}(\mu={mu:.2f},\ \sigma^2={sigma_sq:.2f})$",
    )

    ax.axvline(mu, color="#c53030", linestyle="--", linewidth=1.3, label=rf"Mean Bias $\mu = {mu:.2f}$ dB")
    ax.axvline(0.0, color="#2d3748", linestyle=":", linewidth=1.2, label="Zero Error (0 dB)")

    stats_box = (
        f"M = {m_valid} recordings\n"
        f"Bias (μ) = {mu:+.3f} dB\n"
        f"Variance (σ²) = {sigma_sq:.3f} dB²\n"
        f"Std Dev (σ) = {sigma:.3f} dB\n"
        f"Shapiro-Wilk p = {shapiro_p:.3f}"
    )
    ax.text(
        0.03,
        0.95,
        stats_box,
        transform=ax.transAxes,
        fontsize=9.5,
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="#f7fafc", edgecolor="#cbd5e0"),
    )

    ax.set_title("Pooled Smartphone Measurement Error Distribution vs. Fitted Normal", fontsize=12, fontweight="bold")
    ax.set_xlabel(r"Measurement Error $e_i = L_{\mathrm{phone}, i} - L_{\mathrm{reference}, i}$ (dB)", fontsize=10.5)
    ax.set_ylabel("Probability Density", fontsize=10.5)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(loc="upper right", fontsize=9)
    fig.tight_layout()

    plot_path = analysis_dir / "Pooled_Error_Histogram_vs_Normal.png"
    fig.savefig(plot_path)
    plt.close(fig)

    print("\n=== OVERALL CALIBRATION RESULTS ===")
    for k, v in overall_summary.items():
        print(f"  {k}: {v}")
    print(f"\nOutputs saved to:\n  - {processed_dir}\n  - {analysis_dir}")

    return overall_summary


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Process SoundTrackr calibration recordings and compute overall bias & variance."
    )
    parser.add_argument(
        "--kit-dir",
        type=Path,
        default=Path("SoundTracker_Calibration_Kit"),
        help="Path to the SoundTracker_Calibration_Kit root directory.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=DEFAULT_RANDOM_SEED,
        help="Random seed for reproducible simulation error draws.",
    )
    args, _ = parser.parse_known_args()
    run_pipeline(kit_root=args.kit_dir, random_seed=args.seed)

[INFO] Created template reference log at: SoundTracker_Calibration_Kit\04_Reference_Meter_Exports\Reference_Log.csv
Total scheduled trials: 180
Raw phone CSVs found:   0
Valid paired trials:    0

[ACTION REQUIRED] Fewer than 2 valid paired trials found.
Ensure raw phone exports are saved under 01_Raw_Data/<Condition>/<Phone_ID>/
and actual reference SLM readings are entered in:
  SoundTracker_Calibration_Kit\04_Reference_Meter_Exports\Reference_Log.csv


In [24]:
from pathlib import Path
import numpy as np
import pandas as pd


def generate_synthetic_calibration_data(kit_root: Path, seed: int = 123) -> None:
    """
    Populates SoundTracker_Calibration_Kit with 180 realistic synthetic Collector CSVs
    and a matching Reference_Log.csv so you can test the full analysis pipeline.
    """
    rng = np.random.default_rng(seed)

    # 1. Read the 180 expected filenames from the kit's Excel sheet
    xlsx_path = kit_root / "SoundTracker_Recording_Names.xlsx"
    recordings_df = pd.read_excel(xlsx_path, sheet_name="Recording Names")
    recordings_df["Condition_Folder"] = (
        recordings_df["Filename (copy to rename)"]
        .astype(str)
        .str.extract(r"^([^_]+_[^_]+)")[0]
    )

    # Simulated biases (dB) for each of the 6 experimental conditions
    condition_biases = {
        "E1_65dB": -0.8,    # Quiet, facing speaker
        "E1_80dB": -1.1,    # Baseline 80 dB, facing speaker
        "E1_95dB": -1.4,    # Loud, facing speaker
        "E2_90deg": -2.1,   # Turned 90 degrees
        "E2_180deg": -3.2,  # Turned 180 degrees
        "E3_Pocket": -6.0,  # Inside pocket (heavily muffled)
    }

    # Simulated metadata & minor device offsets for the 6 phones (IP01-IP03, SA01-SA03)
    phone_profiles = {
        "IP01": ("iPhone 13", "iphone-13", "phone-ip01aa", +0.3),
        "IP02": ("iPhone 14", "iphone-14", "phone-ip02bb", -0.2),
        "IP03": ("iPhone 15", "iphone-15", "phone-ip03cc", +0.1),
        "SA01": ("Galaxy S22", "samsung-s22", "phone-sa01dd", -0.5),
        "SA02": ("Galaxy S23", "samsung-s23", "phone-sa02ee", +0.4),
        "SA03": ("Galaxy A54", "samsung-a54", "phone-sa03ff", -0.9),
    }

    ref_rows = []

    for _, row in recordings_df.iterrows():
        fname = row["Filename (copy to rename)"]
        cond = row["Condition_Folder"]
        pid = row["Phone ID"]
        trial = int(row["Trial"])
        target_db = float(row["Target level (dBA)"])

        # Simulate actual reference meter reading (slightly different from target 65/80/95)
        actual_ref_db = round(target_db + rng.normal(0.0, 0.25), 2)

        ref_rows.append(
            {
                "Filename": fname,
                "Condition_Folder": cond,
                "Experiment": row["Experiment"],
                "Target_Level_dBA": int(target_db),
                "Orientation": row["Orientation"],
                "Pocket": row["Pocket"],
                "Phone_ID": pid,
                "Trial": trial,
                "Ref_LAeq_dB": actual_ref_db,
                "Ref_Before_dB": np.nan,
                "Ref_After_dB": np.nan,
                "Source_Export_File": "",
                "Window_Start_UTC": "",
                "Window_End_UTC": "",
                "Notes": "Synthetic test data",
            }
        )

        # Create 17 one-second rows (so we can verify trimming down to 15 seconds)
        model, dclass, bucket, phone_offset = phone_profiles[pid]
        trial_offset = condition_biases[cond] + phone_offset + rng.normal(0.0, 0.4)
        sec_levels = np.round(actual_ref_db + trial_offset + rng.normal(0.0, 0.35, size=17), 2)

        # Plant one maxed-out 120.0 dB row in Trial 1 of E1_95dB to verify clamp removal
        if cond == "E1_95dB" and trial == 1:
            sec_levels[0] = 120.0

        timestamps = pd.date_range(
            "2026-11-10T10:00:01Z", periods=17, freq="1s"
        ).strftime("%Y-%m-%dT%H:%M:%S.000Z")

        # Match the exact 15-column Collector CSV export format
        collector_df = pd.DataFrame(
            {
                "recorded_at": timestamps,
                "event_id": f"calib-{cond.lower()}",
                "contributor_bucket": bucket,
                "lat": 51.454500,
                "lon": -2.587900,
                "accuracy_m": 5.0,
                "laeq_db": sec_levels,
                "lamax_db": np.round(sec_levels + 4.2, 2),
                "integration_seconds": 1,
                "calibration_offset_db": 0,
                "uploaded": True,
                "invite": "lab01",
                "phone_model": model,
                "device_class": dclass,
                "consent_version": "phase3-draft-1",
            }
        )

        out_csv = kit_root / "01_Raw_Data" / cond / pid / fname
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        collector_df.to_csv(out_csv, index=False)

    # Save Reference_Log.csv into 04_Reference_Meter_Exports/
    ref_dir = kit_root / "04_Reference_Meter_Exports"
    ref_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(ref_rows).to_csv(ref_dir / "Reference_Log.csv", index=False)
    print(f"Generated 180 synthetic phone CSVs and Reference_Log.csv in {kit_root}")


# --- Run the test end-to-end ---
kit_folder = Path("SoundTracker_Calibration_Kit")  # Update path if needed
generate_synthetic_calibration_data(kit_folder)
results = run_pipeline(kit_root=kit_folder, random_seed=42)

Generated 180 synthetic phone CSVs and Reference_Log.csv in SoundTracker_Calibration_Kit
Total scheduled trials: 180
Raw phone CSVs found:   180
Valid paired trials:    180

=== OVERALL CALIBRATION RESULTS ===
  valid_recordings_M: 180
  overall_bias_mu_db: -2.5914
  pooled_variance_sigma2_db2: 3.5624
  pooled_sd_sigma_db: 1.8874
  pure_repeatability_variance_db2: 0.165
  pure_repeatability_sd_db: 0.4062
  shapiro_wilk_W: 0.9058
  shapiro_wilk_p: 0.0
  recommended_random_seed: 42

Outputs saved to:
  - SoundTracker_Calibration_Kit\02_Processed_Data
  - SoundTracker_Calibration_Kit\03_Analysis
